# Codebook-only prompting, and how accurate each model actually is

This notebook does two things.

**First**, it shows the *codebook-only* prompt: nothing but the written codebook plus the
illustrative example each code already carries. No human annotations are shown to the model.
This is the honest test of whether the codebook, as written, is enough for a model to apply
it — which is also a useful diagnostic of the codebook itself. Codes with vague or missing
definitions tend to be exactly the codes the model gets wrong, and they are usually the same
ones the two human coders disagreed about.

**Second**, it runs a train/test evaluation on the human-coded round. The 202 adjudicated
units are split in half; few-shot demonstrations may only be drawn from the train half, and
every model is scored on the same held-out test half. The two human coders are scored against
the same adjudicated labels on the same units, which gives the only meaningful yardstick for
a model's number.

### Reading the metrics

Coding is multi-label: a unit carries zero to six codes out of 38. Per-cell accuracy is
therefore useless (predict nothing and you score ~95%). What matters:

| metric | meaning |
|---|---|
| `micro_f1` | overall precision/recall balance across all code assignments — the headline number |
| `macro_f1_present_codes` | unweighted mean F1 over codes that actually occur, so rare codes count as much as common ones |
| `exact_set_match` | fraction of units where the model's code set is *exactly* the gold set — a harsh bar |
| `mean_jaccard` | average overlap between predicted and gold code sets, a softer version of the above |
| `macro_kappa_present_codes` | chance-corrected agreement, directly comparable to the human-human kappas in `reliability-all-rounds.csv` |

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from thematic_ai import RunConfig, annotate_units, evaluate_run, get_backend
from thematic_ai.evaluate import compare_to_human_reliability, human_baseline
from thematic_ai.pipeline import load_workspace, make_system_prompt
from thematic_ai.prompts import render_few_shot, select_few_shot_units

pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 80)

EVAL_DIR = PROJECT_ROOT / "outputs" / "evaluation"
EVAL_DIR.mkdir(parents=True, exist_ok=True)

## 1. The codebook as the model sees it

Every code is rendered with whatever the codebook provides: definition, inclusion criteria,
exclusion criteria and example. Several codes were added mid-analysis and carry no definition
at all — `derogatory remarks`, `Majority offesnive`, `Normal behavior`, `positive content`.
Worth noting which those are before looking at the scores, because the model has nothing to go
on for them beyond the code's name.

In [ ]:
ws = load_workspace(RunConfig(), test_size=0.5)
display(ws.describe().to_frame("n"))

cb = ws.codebook.to_frame()
cb["has_definition"] = cb["definition"].str.len() > 0
cb["has_example"] = cb["example"].str.len() > 0
print("codes with no definition:", cb.loc[~cb["has_definition"], "code"].tolist())
print("codes with no example: ", cb.loc[~cb["has_example"], "code"].tolist())

print("\n" + "=" * 90)
print("CODEBOOK BLOCK OF THE PROMPT")
print("=" * 90)
print(ws.codebook.render(include_examples=True))

## 2. Train/test split

The split is stratified on how many codes a unit carries. Without that, the handful of
four-and-five-code units can all land on one side and the two halves stop being the same task.

Note the denominator: the adjudicated set includes units a human reviewed and legitimately
resolved to *no* codes. Those only exist in `project-export.json` (the CSV lists positive
labels only), and dropping them would quietly reward a model for over-coding.

In [ ]:
split_summary = pd.DataFrame(
    {
        "train": ws.train["n_gold_codes"].value_counts().sort_index(),
        "test": ws.test["n_gold_codes"].value_counts().sort_index(),
    }
).fillna(0).astype(int)
split_summary.index.name = "codes per unit"
display(split_summary)

test_support = (
    pd.Series([c for codes in ws.test["gold_codes"] for c in codes])
    .value_counts()
    .rename("test_support")
)
print(f"{len(test_support)} of {len(ws.codebook)} codes occur in the test split")
print("codes appearing only once or twice in test (their F1 will be very noisy):")
print(test_support[test_support <= 2].index.tolist())
display(test_support.head(15).to_frame())

## 3. The three prompt variants

`codebook_only` is the codebook and nothing else.

`few_shot` appends worked examples chosen greedily from the *train* half so that between them
they cover as many distinct codes as possible — sampling at random would show the model six
variations of "hate speech" and never a rare code.

`calibrated` adds what a codebook structurally cannot say: how often the coders actually
reached for each code, how many codes they put on one justification, which codes they treated
as alternatives rather than companions, and — retrieved per unit — the already-coded
justifications most similar to the one being coded.

That third variant exists because of a specific diagnosis. With the first two, GPT-5.1
recalled 140 of 174 gold codes but *added* 128 more, applying 2.68 codes per unit where the
humans applied 1.74. Only 15% of its false positives were the right theme with the wrong
code; the rest were extra codes the humans simply declined to apply. The failure was
restraint, not comprehension, and restraint is learnable from the training split.

In [ ]:
FEW_SHOT_K = 12

chosen = select_few_shot_units(ws.train, ws.adjudicated, FEW_SHOT_K, seed=RunConfig().seed)
covered = {c for u in chosen for c in ws.gold.set_index("unit_id")["gold_codes"].get(u, [])}
print(f"{len(chosen)} demonstrations covering {len(covered)} distinct codes")
assert set(chosen).isdisjoint(set(ws.test["unit_id"])), "few-shot leaked into the test split"

print("\n" + "=" * 90)
print("WORKED EXAMPLES APPENDED BY THE few_shot VARIANT")
print("=" * 90)
print(render_few_shot(chosen, ws.gold, ws.adjudicated))

for variant in ("codebook_only", "few_shot", "calibrated"):
    prompt = make_system_prompt(ws, RunConfig(prompt_variant=variant, few_shot_k=FEW_SHOT_K))
    print(f"\n{variant}: {len(prompt):,} prompt characters")

# What `calibrated` adds on top, all derived from the train split.
from thematic_ai.calibration import compute_coding_stats
from thematic_ai.pipeline import make_context_builder

stats = compute_coding_stats(ws.train, ws.codebook)
print("\n" + "=" * 90)
print("CALIBRATION BLOCK")
print("=" * 90)
print(stats.render(ws.codebook))

example = ws.test.iloc[0].to_dict()
print("\n" + "=" * 90)
print("RETRIEVED NEIGHBOURS FOR ONE TEST UNIT")
print("=" * 90)
print(f"unit: {example['unit_text']}\ngold: {sorted(ws.gold_labels[example['unit_id']])}\n")
print(make_context_builder(ws, RunConfig(prompt_variant="calibrated"))(example))

## 4. Run the test split

Set `BACKEND` and run the cell; then change `BACKEND` and run it again. Results accumulate in
`RESULTS`, and each run is also written to `outputs/evaluation/` so the two backends can be
compared in section 8 even across separate sessions.

100 test units × 2 variants is roughly 40 minutes on a local Qwen and a couple of minutes on
GPT-5.1. Drop `TEST_LIMIT` to something small for a first pass.

In [ ]:
BACKEND = "ollama"          # "ollama" | "openai"
MODEL = ""                   # "" -> qwen3.5:latest / gpt-5.1
VARIANTS = ("codebook_only", "few_shot", "calibrated")
TEST_LIMIT = 0               # 0 = the whole test split

RESULTS = globals().get("RESULTS", {})   # survives a re-run with the other backend
test_units = ws.test.head(TEST_LIMIT) if TEST_LIMIT else ws.test

for variant in VARIANTS:
    config = RunConfig(
        backend=BACKEND,
        model=MODEL,
        prompt_variant=variant,
        few_shot_k=FEW_SHOT_K,
        run_label="test_split",
    )   # reasoning_effort and retrieved_neighbours keep their tuned defaults
    prompt = make_system_prompt(ws, config)
    backend = get_backend(config)
    print(f"\n{config.backend}:{config.model} | {variant} | {len(test_units)} units")
    print(" ", backend.health_check())

    run = annotate_units(
        test_units, ws.codebook, config, prompt, backend=backend,
        context_builder=make_context_builder(ws, config),
        progress=lambda done, total: print(f"  {done}/{total}", end="\r"),
    )
    run.save(PROJECT_ROOT / "outputs", prefix="eval")
    result = evaluate_run(run, ws.gold_labels, ws.codebook, ws.adjudicated)

    key = (f"{config.backend}:{config.model}", variant)
    RESULTS[key] = {"run": run, "result": result, "config": config}
    print(
        f"\n  micro-F1 {result.overall['micro_f1']:.3f} | "
        f"macro-F1 {result.overall['macro_f1_present_codes']:.3f} | "
        f"exact-set {result.overall['exact_set_match']:.3f} | "
        f"kappa {result.overall['macro_kappa_present_codes']:.3f} | "
        f"{run.n_failed} failed"
    )

## 5. Scores, with the human coders as the yardstick

The two human coders are scored against the adjudicated labels on the same test units, which
is what makes a model's number readable: 0.60 micro-F1 next to humans at 0.75 means the model
is doing most of what a trained coder does; the same 0.60 next to humans at 0.95 does not.

**Read the two human rows carefully on this dataset.** Adjudication tracked one coder
(Alireza) almost exactly — precision 0.997 and recall 0.965 against the adjudicated labels
across all 201 units, versus roughly 0.39 / 0.42 for the other coder. So "agreement with the
adjudicated gold" here is very close to "agreement with Alireza", and his ~0.99 is not an
independent ceiling. The realistic bar for a second, independent coder is the *lower* human
row. The next cell therefore also scores the model against each coder separately, and against
the union and intersection of the two, so the conclusion does not rest on one adjudicator's
choices.

In [ ]:
test_ids = list(test_units["unit_id"])

rows = human_baseline(ws.annotations, ws.codebook, ws.gold_labels, test_ids).to_dict("records")
for (annotator, variant), entry in RESULTS.items():
    rows.append(
        {
            "annotator": annotator,
            "kind": "model",
            "prompt_variant": variant,
            **entry["result"].overall,
            **{f"span_{k}": v for k, v in entry["result"].spans.items()},
        }
    )

summary = pd.DataFrame(rows)
headline = [
    "annotator", "kind", "prompt_variant", "n_units",
    "micro_f1", "macro_f1_present_codes", "macro_kappa_present_codes",
    "exact_set_match", "mean_jaccard", "at_least_one_correct",
    "gold_labels", "pred_labels",
]
display(summary[[c for c in headline if c in summary.columns]].round(3))
summary.to_csv(EVAL_DIR / "summary__notebook.csv", index=False)

models = summary[summary["kind"] == "model"]
if len(models):
    fig, ax = plt.subplots(figsize=(11, 4.5))
    labels = [f"{r.annotator}\n{r.prompt_variant}" for r in models.itertuples()]
    x = np.arange(len(labels))
    for offset, metric, colour in [
        (-0.25, "micro_f1", "#3b5f9e"),
        (0.0, "macro_f1_present_codes", "#b5623a"),
        (0.25, "exact_set_match", "#4f9b7f"),
    ]:
        ax.bar(x + offset, models[metric], width=0.25, label=metric, color=colour)
    for human in summary[summary["kind"] == "human"].itertuples():
        ax.axhline(human.micro_f1, ls="--", lw=1, color="grey")
        ax.text(len(labels) - 0.4, human.micro_f1 + 0.01, f"human {human.annotator}", fontsize=8, color="grey")
    ax.set_xticks(x, labels, fontsize=9)
    ax.set_ylim(0, 1)
    ax.set_ylabel("score on the held-out test split")
    ax.legend(fontsize=8)
    ax.set_title("model accuracy against adjudicated human coding")
    fig.tight_layout()
    plt.show()

# Robustness: score the same predictions against each coder individually, and
# against the union and intersection of the two, rather than only the gold.
from thematic_ai.data import gold_label_sets, label_matrix
from thematic_ai.evaluate import agreement_with_each_coder, overall_metrics

alt_rows = []
for (annotator, variant), entry in RESULTS.items():
    per_coder = agreement_with_each_coder(entry["run"], ws.annotations, ws.codebook, test_ids)
    per_coder.insert(0, "prompt_variant", variant)
    per_coder.insert(0, "model", annotator)
    alt_rows.append(per_coder)

    for source in ("union", "intersection"):
        labels = gold_label_sets(ws.adjudicated, ws.annotations, source)
        scored = [u for u in test_ids if u in labels]
        reference = label_matrix(labels, scored, ws.codebook.names)
        prediction = label_matrix(entry["run"].label_sets, scored, ws.codebook.names)
        alt_rows.append(
            pd.DataFrame([{
                "model": annotator, "prompt_variant": variant,
                "reference": f"{source} of both coders", "kind": "alt_gold",
                **overall_metrics(reference, prediction),
            }])
        )

if alt_rows:
    alt = pd.concat(alt_rows, ignore_index=True)
    display(alt[["model", "prompt_variant", "reference", "n_units", "micro_precision",
                 "micro_recall", "micro_f1", "mean_jaccard"]].round(3))

## 6. Per-code and per-theme

Micro-F1 hides everything interesting. Per-code scores show *which* distinctions the model
can make, and the theme rollup shows how much of the error is a model picking the wrong code
within the right theme — for instance `Death wish` versus `Threats of violence`, a boundary
the two human coders also blurred.

In [ ]:
INSPECT = list(RESULTS)[-1]      # (annotator, variant) — change to look at another run
result = RESULTS[INSPECT]["result"]
print("inspecting:", INSPECT)

per_code = result.per_code[result.per_code["support_gold"] > 0].copy()
display(per_code[["code", "support_gold", "support_pred", "tp", "fp", "fn", "precision", "recall", "f1", "cohens_kappa"]].round(3))
display(result.per_theme[["theme", "support_gold", "support_pred", "precision", "recall", "f1", "cohens_kappa"]].round(3))

fig, ax = plt.subplots(figsize=(9, max(4, 0.28 * len(per_code))))
ordered = per_code.sort_values("f1")
ax.barh(ordered["code"], ordered["f1"], color="#3b5f9e")
for y, (f1, support) in enumerate(zip(ordered["f1"], ordered["support_gold"])):
    ax.text(min(f1 + 0.01, 0.97), y, f"n={support}", va="center", fontsize=7, color="grey")
ax.set_xlim(0, 1)
ax.set_xlabel("F1 against adjudicated gold")
ax.set_title(f"per-code accuracy — {INSPECT[0]} ({INSPECT[1]})")
ax.tick_params(labelsize=8)
fig.tight_layout()
plt.show()

print("\ncodes the model never predicted but gold uses:",
      per_code.loc[per_code["support_pred"] == 0, "code"].tolist())
print("codes the model over-applies (pred > 2x gold):",
      per_code.loc[per_code["support_pred"] > 2 * per_code["support_gold"], "code"].tolist())

# Both models here recall far more than they get right, i.e. they over-code.
# Re-scoring the predictions already made, with hedged codes discarded, is the
# cheapest accuracy available: no new API calls, just a threshold.
from thematic_ai.evaluate import confidence_sweep

sweep = confidence_sweep(RESULTS[INSPECT]["run"], ws.gold_labels, ws.codebook)
display(sweep[["min_confidence", "pred_labels", "micro_precision", "micro_recall",
               "micro_f1", "exact_set_match", "mean_jaccard"]].round(3))

best = sweep.loc[sweep["micro_f1"].idxmax()]
print(f"best threshold {best['min_confidence']:.2f}: micro-F1 {best['micro_f1']:.3f} "
      f"vs {sweep.iloc[0]['micro_f1']:.3f} unfiltered")
print(f"apply it on the full corpus with --min-confidence {best['min_confidence']:.2f}")

## 7. Model agreement vs human-human agreement

`data/reliability-all-rounds.csv` records the Cohen's kappa the two human coders reached on
each code. Putting the model's kappa beside it separates two very different failure modes: a
code where the model is weak *and the humans agreed* is a model problem, while a code where
both are weak is a codebook problem, and no amount of prompting will fix it.

In [ ]:
comparison = compare_to_human_reliability(
    result.per_code, str(PROJECT_ROOT / "data" / "reliability-all-rounds.csv")
)
comparison = comparison[comparison["support_gold"] > 0]
display(
    comparison[["code", "support_gold", "cohens_kappa", "human_human_kappa", "kappa_gap"]]
    .sort_values("kappa_gap")
    .round(3)
)

fig, ax = plt.subplots(figsize=(6.5, 6.5))
ax.scatter(comparison["human_human_kappa"], comparison["cohens_kappa"], s=comparison["support_gold"] * 6, alpha=0.6)
lims = (-0.1, 1.0)
ax.plot(lims, lims, ls="--", color="grey", lw=1)
for row in comparison.itertuples():
    if pd.notna(row.human_human_kappa) and abs(row.kappa_gap) > 0.25:
        ax.annotate(row.code, (row.human_human_kappa, row.cohens_kappa), fontsize=7, alpha=0.8)
ax.set_xlim(*lims)
ax.set_ylim(*lims)
ax.set_xlabel("human vs human kappa (round 1)")
ax.set_ylabel(f"{INSPECT[0]} vs adjudicated kappa")
ax.set_title("below the line: the model is worse than the humans were with each other")
fig.tight_layout()
plt.show()

print("model beats human-human agreement on:",
      comparison.loc[comparison["kappa_gap"] > 0, "code"].tolist())

# The decisive question: is the remaining error the model's, or the codebook's?
from thematic_ai.evaluate import ceiling_analysis

ceiling = ceiling_analysis(result.per_code, str(PROJECT_ROOT / "data" / "reliability-all-rounds.csv"))
detail = ceiling.pop("detail")
print("\n" + pd.Series(ceiling).round(3).to_string())
print(
    f"\nThe model scores {ceiling['f1_where_humans_agreed']:.2f} on codes the two coders agreed "
    f"about and {ceiling['f1_where_humans_disagreed']:.2f} on codes they did not "
    f"(correlation r={ceiling['kappa_f1_correlation']:.2f}). The closer that correlation is to 1, "
    f"the more of the remaining error belongs to the codebook rather than the model."
)

# Theme-level: did it at least identify the right kind of reasoning?
theme_tp = result.per_theme["tp"].sum()
theme_f1 = 2 * theme_tp / (2 * theme_tp + result.per_theme["fp"].sum() + result.per_theme["fn"].sum())
print(f"\ntheme-level micro-F1: {theme_f1:.3f}  (vs code-level {result.overall['micro_f1']:.3f})")

# What a pruned codebook would buy, if you are willing to merge the unreliable codes.
rel = pd.read_csv(PROJECT_ROOT / "data" / "reliability-all-rounds.csv")
rel["k"] = pd.to_numeric(rel["cohens_kappa"].astype(str).str.lstrip("'"), errors="coerce")
scored_units = list(result.gold_matrix.index)
predictions = RESULTS[INSPECT]["run"].label_sets
from thematic_ai.evaluate import overall_metrics

pruned = []
for floor in (0.0, 0.2, 0.3, 0.4, 0.5):
    keep = set(rel.loc[rel["k"] >= floor, "code"])
    codes = [c for c in ws.codebook.names if c in keep]
    if not codes:
        continue
    g = label_matrix({u: ws.gold_labels[u] & keep for u in scored_units}, scored_units, codes)
    p = label_matrix({u: predictions.get(u, set()) & keep for u in scored_units}, scored_units, codes)
    pruned.append({"min_human_kappa": floor, "n_codes": len(codes), **overall_metrics(g, p)})
display(pd.DataFrame(pruned)[["min_human_kappa", "n_codes", "gold_labels", "micro_f1", "exact_set_match"]].round(3))

## 8. Read the actual mistakes

Aggregate metrics tell you how much is wrong, never what. Read twenty of these before
trusting or rejecting a model — a good share of "errors" on this dataset turn out to be
defensible codings that simply differ from the adjudicator's choice, which is itself a finding
worth reporting.

In [ ]:
errors = result.errors
print(f"{len(errors)} of {result.overall['n_units']} test units differ from gold\n")

for row in errors.sort_values("n_agreed").head(10).itertuples():
    print(f"— {row.unit_text}")
    print(f"    human: {row.gold_codes}")
    print(f"    model: {row.pred_codes}")
    if row.missed:
        print(f"    missed:   {row.missed}")
    if row.spurious:
        print(f"    spurious: {row.spurious}")
    print()

errors.to_csv(EVAL_DIR / f"errors__{INSPECT[0].replace(':', '-')}__{INSPECT[1]}.csv", index=False)

print("most frequently missed codes:")
print(pd.Series([c for s in errors["missed"] if s for c in s.split("; ")]).value_counts().head(8).to_string())
print("\nmost frequently spurious codes:")
print(pd.Series([c for s in errors["spurious"] if s for c in s.split("; ")]).value_counts().head(8).to_string())

## 9. GPT-5.1 vs Qwen, side by side

Run section 4 once per backend — in this session or an earlier one — then this cell collects
every summary written to `outputs/evaluation/` and puts them in one table. Do not compare runs
whose `n_units` differ; that means one of them was capped by `TEST_LIMIT`.

The equivalent from a terminal:

```bash
python scripts/run_experiment.py --backend ollama --model qwen3.5:latest
python scripts/run_experiment.py --backend openai --model gpt-5.1
```

In [ ]:
frames = [pd.read_csv(path) for path in sorted(EVAL_DIR.glob("summary__*.csv"))]
if not frames:
    print("no summaries yet — run section 4 for at least one backend")
else:
    everything = pd.concat(frames, ignore_index=True)
    everything = everything.drop_duplicates(subset=["annotator", "kind", "prompt_variant", "n_units"], keep="last")
    columns = [
        "annotator", "kind", "prompt_variant", "n_units",
        "micro_precision", "micro_recall", "micro_f1",
        "macro_f1_present_codes", "macro_kappa_present_codes",
        "exact_set_match", "mean_jaccard",
    ]
    table = everything[[c for c in columns if c in everything.columns]].round(3)
    display(table.sort_values(["kind", "micro_f1"], ascending=[True, False]))

    if table["n_units"].nunique() > 1:
        print("\nwarning: runs cover different numbers of units and are not directly comparable")

    picks = table[table["kind"] == "model"]
    if len(picks) > 1:
        best = picks.loc[picks["micro_f1"].idxmax()]
        print(f"\nbest configuration: {best['annotator']} with the {best['prompt_variant']} prompt "
              f"(micro-F1 {best['micro_f1']:.3f})")
        print("use it on the rest of the corpus with:")
        backend_name, model_name = best["annotator"].split(":", 1)
        print(f"  python scripts/run_annotate.py --backend {backend_name} --model {model_name} "
              f"--prompt-variant {best['prompt_variant']} --units unlabelled")